In [2]:
import torch
import torch.nn as nn

device = torch.device("mps")
print(f"using device: {device}")

using device: mps


In [3]:
# Toy dimensions

L = 64 # sequence length
N_seq = 8 # number of seq in MSA
c_m = 64 # MSA representation dims
c_z = 32 # pair representation dims

print (f"MSA repr shape: ({N_seq}, {L}, {c_m})")
print (f"Pair repr shape: ({L}, {L}, {c_z})")

MSA repr shape: (8, 64, 64)
Pair repr shape: (64, 64, 32)


In [4]:
aatype = torch.randint(0, 21, (L,), device=device)
residue_index = torch.arange(L, device=device)
msa = torch.randint(0, 23, (N_seq, L), device=device)

print("aatype:        ", aatype.shape)       # (L,)
print("residue_index: ", residue_index.shape) # (L,)
print("msa:           ", msa.shape)           # (N_seq, L)


aatype:         torch.Size([64])
residue_index:  torch.Size([64])
msa:            torch.Size([8, 64])


In [5]:
print(residue_index)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63], device='mps:0')


In [6]:
import torch.nn.functional as F

msa_one_hot = F.one_hot(msa, num_classes=23).float() # (N_seq, L, 23)
print("msa_one_hot:", msa_one_hot.shape)


msa_one_hot: torch.Size([8, 64, 23])


In [7]:
msa

tensor([[ 2,  0,  8, 11, 13,  9, 16, 13, 18, 19, 13, 12, 22, 14,  6, 16, 14, 15,
          1,  2,  8,  2, 16,  9,  9, 21,  0, 16,  9, 12, 22, 14, 15, 14,  1, 21,
         18,  4, 13, 17,  5, 18, 12, 21, 17, 14,  6, 18,  3, 19,  7, 15, 20,  9,
         12,  2, 11, 13,  2, 11, 10,  6,  2,  8],
        [19,  5,  1, 12,  6, 14, 14, 14, 21,  6,  0,  6, 21, 22, 16,  7,  5,  8,
         17,  4, 18, 18, 16,  8,  8,  2,  8, 17, 15, 17, 12,  6, 14,  6, 16,  3,
          0, 14, 10, 17, 12,  5, 14, 10, 13, 20,  4, 14,  9, 15, 18, 20, 18, 14,
          9, 16, 19,  4,  7, 16,  4, 19, 19,  8],
        [19,  7,  5, 19, 19,  0, 20,  0,  0, 22,  4, 16, 16,  5, 14,  8, 17, 13,
         10,  7, 18,  0, 21, 13,  9,  5, 14, 16, 20, 19, 20, 22, 22,  2, 16, 20,
         21, 15, 16,  2,  9,  0,  3, 14, 20,  7,  4, 16,  0,  3, 17,  7, 19,  2,
         19, 17, 21, 16, 15, 20,  7,  1, 16, 13],
        [15, 19,  5,  2, 17,  9, 20, 13, 11, 10,  6, 22, 10,  7,  7,  0, 10, 21,
         12,  1,  6,  0, 12,  4, 18,  4,

In [8]:
class InputEmbedder(nn.Module):
    def __init__(self, c_m, c_z):
        super().__init__()
        self.msa_embedder = nn.Linear(23, c_m)
        self.pair_embedder = nn.Linear(65, c_z)

    def forward(self, msa, residue_index):
        # msa: (N_seq, L), residue_index: (L,)

        #MSA
        msa_one_hot = F.one_hot(msa, num_classes=23).float()
        m = self.msa_embedder(msa_one_hot)

        #Pair
        rel_pos = residue_index[None, :] - residue_index[:,None] #(L, L)
        rel_pos_one_hot = F.one_hot(rel_pos.clamp(-32, 32) + 32, num_classes=65).float()
        z = self.pair_embedder(rel_pos_one_hot)

        return m, z
    

embedder = InputEmbedder(c_m, c_z).to(device)
m, z = embedder(msa, residue_index)
print("m:", m.shape)
print("z:", z.shape)

m: torch.Size([8, 64, 64])
z: torch.Size([64, 64, 32])


In [9]:
c_head = 8 # dimensions per attention head
num_heads = 4

to_q = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)
to_k = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)
to_v = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)

pair_bias = nn.Linear(c_z, num_heads, bias=False).to(device)

row = m[0]
q = to_q(row)
k = to_k(row)
v = to_v(row)

print("q:", q.shape)
print("Pair bias input z:", z.shape)
print("Pair bias output:", pair_bias(z).shape)

q: torch.Size([64, 32])
Pair bias input z: torch.Size([64, 64, 32])
Pair bias output: torch.Size([64, 64, 4])


In [10]:
pair_bias

Linear(in_features=32, out_features=4, bias=False)

In [11]:
from einops import rearrange

q = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)
k = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)
v = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)

print('q:', q.shape)

# Raw attention scores: for each head, dot every query with every key
scale = c_head ** - 0.5
scores = torch.einsum('h i d, h j d -> h i j', q, k) * scale
print("scores:", scores.shape)

#pair bias
bias = rearrange(pair_bias(z), 'i j h -> h i j')
print("bias:", bias.shape)

scores = scores + bias

q: torch.Size([4, 64, 8])
scores: torch.Size([4, 64, 64])
bias: torch.Size([4, 64, 64])


In [12]:
# softmax over the j axis, for each query pos i, weights over all key positions, j to sum to 1
attn_weights = torch.softmax(scores, dim=-1)
print("attn_weights:", attn_weights.shape)
print("row sums (should all be 1.0):", attn_weights[0, 0].sum().item())


# aggregate values: for each position i, weighted sum of all value vectors
out = torch.einsum('h i j, h j d -> h i d', attn_weights, v)
print("out:", out.shape) # (L, num_heads*c_head)

out = rearrange(out, 'h l d -> l (h d)')
print("out merged:", out.shape)

# final linear projection back to c_m
to_out = nn.Linear(num_heads * c_head, c_m).to(device)
out = to_out(out)
print("out projected:", out.shape)

attn_weights: torch.Size([4, 64, 64])
row sums (should all be 1.0): 1.000000238418579
out: torch.Size([4, 64, 8])
out merged: torch.Size([64, 32])
out projected: torch.Size([64, 64])


In [13]:
to_gate = nn.Linear(c_m, num_heads * c_head).to(device)

gate = torch.sigmoid(to_gate(row))
gate = rearrange(gate, 'l (h d) -> h l d', h = num_heads)

# recompute out before merging
out_heads = torch.einsum('h i j, h j d -> h i d', attn_weights, v)

# apply gate element wise
out_gated = gate * out_heads

# merge and project
out_gated = rearrange(out_gated, 'h l d -> l (h d)')
out_gated = to_out(out_gated)
print('gated output:', out_gated.shape)

gated output: torch.Size([64, 64])


In [14]:
class MSARowAttention(nn.Module):
    def __init__(self, c_m, c_z, c_head, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_m)
        self.to_q = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_k = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_v = nn.Linear(c_m, c_head * num_heads, bias=False)

        self.to_gate = nn.Linear(c_m, c_head * num_heads)
        self.pair_bias = nn.Linear(c_z, num_heads, bias=False)
        self.to_out = nn.Linear(c_head * num_heads, c_m)

    def forward(self, m, z):
        # m: (N_seq, L, c_m), z: (L, L, c_z)
        m = self.norm(m)
        scale = self.c_head ** -0.5
        bias = rearrange(self.pair_bias(z), 'i j h -> h i j') # (H, L, L)

        q = rearrange(self.to_q(m), 'n l (h d) -> h n l d', h = self.num_heads)
        k = rearrange(self.to_k(m), 'n l (h d) -> h n l d', h = self.num_heads)
        v = rearrange(self.to_v(m), 'n l (h d) -> h n l d', h = self.num_heads)

        scores = torch.einsum('h n i d, h n j d -> h n i j', q, k) * scale
        scores = scores + bias.unsqueeze(1) # broadcast over N_seq
        attn = torch.softmax(scores, dim=-1) # (H, N_seq, L, L)

        out = torch.einsum('h n i j, h n j d -> h n i d', attn, v)
        gate = torch.sigmoid(rearrange(self.to_gate(m), 'n l (h d) -> h n l d', h = self.num_heads))
        out = rearrange(gate * out, 'h n l d -> n l (h d)')
        return self.to_out(out) # (N_seq, L, c_m)

row_attn = MSARowAttention(c_m, c_z, c_head=8, num_heads=4).to(device)
out = row_attn(m,z)
print('output:', out.shape)

output: torch.Size([8, 64, 64])


In [15]:
m_new = m + row_attn(m,z)
print("m_new:", m_new.shape)

m_new: torch.Size([8, 64, 64])


In [16]:

class MSAColumnAttention(nn.Module):
    def __init__(self, c_m, c_head, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_m)
        self.to_q = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_k = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_v = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_gate = nn.Linear(c_m, c_head * num_heads)
        self.to_out = nn.Linear(c_head * num_heads, c_m)

    def forward(self, m):
        # m: (N_seq, L, c_m)
        m = self.norm(m)
        scale = self.c_head **-0.5

        # transpose so L is the outer dim and attend across N_seq for each position
        m_t = rearrange(m, 'n l d -> l n d') # (L, N_seq, c_m)

        q = rearrange(self.to_q(m_t), 'l n (h d) -> h l n d', h = self.num_heads)
        k = rearrange(self.to_k(m_t), 'l n (h d) -> h l n d', h = self.num_heads)
        v = rearrange(self.to_v(m_t), 'l n (h d) -> h l n d', h = self.num_heads)

        scores = torch.einsum('h l i d, h l j d -> h l i j', q, k) * scale
        attn = torch.softmax(scores, dim=-1) # H, L, N_seq, N_seq

        out = torch.einsum('h l i j, h l j d -> h l i d', attn, v)
        gate = torch.sigmoid(rearrange(self.to_gate(m_t), 'l n (h d) -> h l n d', h=self.num_heads))
        out = rearrange(gate * out, 'h l n d -> n l (h d)')
        return self.to_out(out)

col_attn = MSAColumnAttention(c_m, c_head=8, num_heads=4).to(device)
m_new = m + col_attn(m)
print('m_new:', m_new.shape) # 8, 64, 64

m_new: torch.Size([8, 64, 64])


In [17]:
class MSATransition(nn.Module):
    def __init__(self, c_m, expansion=4):
        super().__init__()
        self.norm = nn.LayerNorm(c_m)
        self.ff = nn.Sequential(
            nn.Linear(c_m, c_m * expansion),
            nn.ReLU(),
            nn.Linear(c_m * expansion, c_m)
        )

    def forward(self, m):
        return self.ff(self.norm(m)) # N_seq, L, c_m

transition = MSATransition(c_m).to(device)
m_new = m + transition(m)
print("m_new:", m_new.shape)   # (8, 64, 64)

m_new: torch.Size([8, 64, 64])


In [18]:
c_hidden_opm = 16 # project down before outer product

linear_a = nn.Linear(c_m, c_hidden_opm).to(device)
linear_b = nn.Linear(c_m, c_hidden_opm).to(device)

a = linear_a(m) # N_seq, L, c_hidden_opm
b = linear_b(m)

print("a:", a.shape)

a: torch.Size([8, 64, 16])


In [19]:
# outer product: for every pair(i, j), combine a[:,i,:] with b[:, j, :]
# einsum for each sequence s, position i, position j, outer product of a[s, i, and b[s, j]

outer = torch.einsum('s i d, s j e -> s i j d e', a, b) # n_seq, L, L, c_hidden, c_hidden
print("outer:", outer.shape)

# mean over sequences, covarying patterns survive
outer_mean = outer.mean(dim=0)
print("outer_mean:", outer_mean.shape)



outer: torch.Size([8, 64, 64, 16, 16])
outer_mean: torch.Size([64, 64, 16, 16])


In [20]:
class OuterProductMean(nn.Module):
    def __init__(self, c_m, c_z, c_hidden=16):
        super().__init__()
        self.norm = nn.LayerNorm(c_m)
        self.linear_a = nn.Linear(c_m, c_hidden)
        self.linear_b = nn.Linear(c_m, c_hidden)
        self.linear_out = nn.Linear(c_hidden*c_hidden, c_z)

    def forward(self, m, z):
        m = self.norm(m)
        a = self.linear_a(m)
        b = self.linear_b(m)
        outer = torch.einsum('s i d, s j e -> s i j d e', a, b).mean(dim=0)
        z = z + self.linear_out(rearrange(outer, 'i j d e -> i j (d e)'))
        return z

opm = OuterProductMean(c_m, c_z).to(device)
z_new = opm(m, z)
print("z_new:", z_new.shape)   # (64, 64, 32)

z_new: torch.Size([64, 64, 32])


In [21]:
z

tensor([[[ 0.1318,  0.0899, -0.1428,  ...,  0.0704, -0.0527, -0.1771],
         [ 0.1558,  0.0467, -0.0126,  ...,  0.0423,  0.0403, -0.0673],
         [-0.0385,  0.0302, -0.1425,  ...,  0.0199, -0.1349, -0.1403],
         ...,
         [ 0.1921,  0.1119, -0.1809,  ..., -0.0114, -0.0608, -0.0848],
         [ 0.1921,  0.1119, -0.1809,  ..., -0.0114, -0.0608, -0.0848],
         [ 0.1921,  0.1119, -0.1809,  ..., -0.0114, -0.0608, -0.0848]],

        [[-0.0003,  0.1431, -0.2296,  ...,  0.0113, -0.0578, -0.0066],
         [ 0.1318,  0.0899, -0.1428,  ...,  0.0704, -0.0527, -0.1771],
         [ 0.1558,  0.0467, -0.0126,  ...,  0.0423,  0.0403, -0.0673],
         ...,
         [ 0.1921,  0.1119, -0.1809,  ..., -0.0114, -0.0608, -0.0848],
         [ 0.1921,  0.1119, -0.1809,  ..., -0.0114, -0.0608, -0.0848],
         [ 0.1921,  0.1119, -0.1809,  ..., -0.0114, -0.0608, -0.0848]],

        [[-0.0243,  0.0134, -0.2006,  ..., -0.0080,  0.0752, -0.1471],
         [-0.0003,  0.1431, -0.2296,  ...,  0

In [22]:
c_hidden_tri = 32

linear_a = nn.Linear(c_z, c_hidden_tri).to(device)
linear_b = nn.Linear(c_z, c_hidden_tri).to(device)
linear_g = nn.Linear(c_z, c_hidden_tri).to(device) # gate

a = linear_a(z_new)
b = linear_b(z_new)
g = torch.sigmoid(linear_g(z_new))

print("a:", a.shape)
print("g:", g.shape)

a: torch.Size([64, 64, 32])
g: torch.Size([64, 64, 32])


In [23]:
class TriangularMultiplactiveUpdate(nn.Module):
    def __init__(self, c_z, c_hidden, direction='outgoing'):
        super().__init__()
        assert direction in ('outgoing', 'incoming')
        self.direction = direction
        self.norm = nn.LayerNorm(c_z)
        self.linear_a = nn.Linear(c_z, c_hidden)
        self.linear_b = nn.Linear(c_z, c_hidden)
        self.linear_g = nn.Linear(c_z, c_hidden)
        self.linear_out = nn.Linear(c_hidden, c_z)
        self.norm_out = nn.LayerNorm(c_hidden)

    def forward(self, z):
        # z (L, L, c_z)
        z = self.norm(z)
        a = self.linear_a(z)
        b = self.linear_b(z)
        g = torch.sigmoid(self.linear_g(z))

        if self.direction == 'outgoing':
            # z[i, k] and z[j, k] -> update z[i,j]
            triangle = torch.einsum('i k d, j k d -> i j d', a, b)
        else:
            # z[k, i] and z[k, j] -> update z[i,j]
            triangle = torch.einsum('k i d, k j d -> i j d', a, b)
        return g * self.linear_out(torch.sigmoid(self.norm_out(triangle)))

tri_out = TriangularMultiplactiveUpdate(c_z, c_hidden=32, direction='outgoing').to(device)
tri_in = TriangularMultiplactiveUpdate(c_z, c_hidden=32, direction='incoming').to(device)

z = z_new + tri_out(z_new)
z = z + tri_in(z)
print("z: ", z.shape) # 64, 64, 32

z:  torch.Size([64, 64, 32])


In [24]:
z_new.device


device(type='mps', index=0)

In [25]:
z.device

device(type='mps', index=0)

In [26]:
c_head_tri = 16
num_heads_tri = 16

norm = nn.LayerNorm(c_z).to(device)
linear_q = nn.Linear(c_z, c_head_tri *num_heads_tri, bias=False).to(device)
linear_k = nn.Linear(c_z, c_head_tri *num_heads_tri, bias=False).to(device)
linear_v = nn.Linear(c_z, c_head_tri *num_heads_tri, bias=False).to(device)
linear_b = nn.Linear(c_z, num_heads_tri, bias=False).to(device)
linear_g = nn.Linear(c_z, c_head_tri, bias=False).to(device)

z_norm = norm(z)

q = rearrange(linear_q(z_norm), 'i j (h d) -> i h j d', h=num_heads_tri)
k = rearrange(linear_k(z_norm), 'i j (h d) -> i h j d', h=num_heads_tri)
v = rearrange(linear_v(z_norm), 'i j (h d) -> i h j d', h=num_heads_tri)

print("q: ", q.shape) # L num_heads L c_head

bias = rearrange(linear_b(z_norm), 'j k h -> h j k')
print("bias:", bias.shape) # num_heads L L

q:  torch.Size([64, 16, 64, 16])
bias: torch.Size([16, 64, 64])


In [27]:
scale = c_head_tri ** -0.5

# attention scores for each starting node i, head h, scores between positions j and k
scores = torch.einsum('i h j d, i h k d -> i h j k', q, k) * scale
scores = scores + bias.unsqueeze(0)
attn = torch.softmax(scores, dim=-1)

#aggregate
out = torch.einsum('i h j k, i h k d -> i h j d', attn, v) #L H L c_head

# gate and project
g = torch.sigmoid(rearrange(linear_g(z_norm), 'i j (h d) -> i h j d', h=num_heads_tri))
out = rearrange(g * out, 'i h j d -> i j (h d)') #L L H * c_head

linear_out = nn.Linear(num_heads_tri * c_head_tri, c_z).to(device)
delta_z = linear_out(out)
z = z+delta_z

print("z:", z.shape)

z: torch.Size([64, 64, 32])


In [28]:
class TriangularSelfAttention(nn.Module):
    def __init__(self, c_z, c_head, num_heads, direction='starting'):
        super().__init__()
        assert direction in ('starting', 'ending')
        self.direction = direction
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_z)
        self.linear_q = nn.Linear(c_z, c_head * num_heads, bias=False)
        self.linear_k = nn.Linear(c_z, c_head * num_heads, bias=False)
        self.linear_v = nn.Linear(c_z, c_head * num_heads, bias=False)
        self.linear_b = nn.Linear(c_z, num_heads, bias=False)
        self.linear_g = nn.Linear(c_z, c_head * num_heads)
        self.linear_out = nn.Linear(c_head * num_heads, c_z)

    def forward(self, z):
        if self.direction == 'ending':
            z = z.transpose(0,1)

        z_norm = self.norm(z)
        q = rearrange(self.linear_q(z_norm), 'i j (h d) -> i h j d', h=self.num_heads)
        k = rearrange(self.linear_k(z_norm), 'i j (h d) -> i h j d', h=self.num_heads)
        v = rearrange(self.linear_v(z_norm), 'i j (h d) -> i h j d', h=self.num_heads)
        bias = rearrange(self.linear_b(z_norm), 'j k h -> h j k')

        scores = torch.einsum('i h j d, i h k d -> i h j k', q, k) * (self.c_head**-0.5)
        scores = scores + bias.unsqueeze(0)
        attn = torch.softmax(scores, dim=-1)

        out = torch.einsum('i h j k, i h k d -> i h j d', attn, v)
        g = torch.sigmoid(rearrange(self.linear_g(z_norm), 'i j (h d) -> i h j d', h=self.num_heads))
        out = self.linear_out(rearrange(g * out, 'i h j d -> i j (h d)'))

        if self.direction == 'ending':
            out = out.transpose(0,1)
        return out

tri_att_start = TriangularSelfAttention(c_z, c_head=16, num_heads=4, direction='starting').to(device)
tri_att_end = TriangularSelfAttention(c_z, c_head=16, num_heads=4, direction='ending').to(device)

z = z+tri_att_start(z)
z = z + tri_att_end(z)
print('z:', z.shape)


z: torch.Size([64, 64, 32])


In [29]:
class PairTransition(nn.Module):
    def __init__(self, c_z, expansion=4):
        super().__init__()
        self.norm = nn.LayerNorm(c_z)
        self.ff = nn.Sequential(
            nn.Linear(c_z, c_z * expansion),
            nn.ReLU(),
            nn.Linear(c_z * expansion, c_z)
        )

    def forward(self, z):
        return self.ff(self.norm(z))

class EvoformerBloc(nn.Module):
    def __init__(self, c_m, c_z):
        super().__init__()
        self.row_att = MSARowAttention(c_m, c_z, c_head=8, num_heads=4)
        self.col_att = MSAColumnAttention(c_m, c_head=8, num_heads=4)
        self.msa_tran = MSATransition(c_m)
        self.opm = OuterProductMean(c_m, c_z)
        self.tri_mul_out = TriangularMultiplactiveUpdate(c_z, c_hidden=32, direction='outgoing')
        self.tri_mul_in = TriangularMultiplactiveUpdate(c_z, c_hidden=32, direction='incoming')
        self.tri_att_start = TriangularSelfAttention(c_z, c_head=16, num_heads=4, direction='starting')
        self.tri_att_end = TriangularSelfAttention(c_z, c_head=16, num_heads=4, direction='ending')
        self.pair_tran = PairTransition(c_z)

    def forward(self, m, z):
        m = m + self.row_att(m, z)
        m = m+ self.col_att(m)
        m = m+ self.msa_tran(m)
        z = self.opm(m, z)
        z = z + self.tri_mul_out(z)
        z = z + self.tri_mul_in(z)
        z = z + self.tri_att_start(z)
        z = z + self.tri_att_end(z)
        z = z + self.pair_tran(z)
        return m, z

block = EvoformerBloc(c_m, c_z).to(device)
m_out, z_out = block(m, z)
print("m_out:", m_out.shape)
print('z_out:', z_out.shape)


m_out: torch.Size([8, 64, 64])
z_out: torch.Size([64, 64, 32])


In [30]:
class EvoformerStack(nn.Module):
    def __init__(self, c_m,  c_z, num_blocks):
        super().__init__()
        self.blocks = nn.ModuleList([EvoformerBloc(c_m, c_z) for _ in range(num_blocks)])
        self.norm_m = nn.LayerNorm(c_m)
        self.norm_z = nn.LayerNorm(c_z)

    def forward(self, m, z):
        for block in self.blocks:
            m, z = block(m, z)
        return self.norm_m(m), self.norm_z(z)

evoformer = EvoformerStack(c_m, c_z, num_blocks=4).to(device)
m_out, z_out = evoformer(m, z)
print("m_out:", m_out.shape)
print("z_out:", z_out.shape)

m_out: torch.Size([8, 64, 64])
z_out: torch.Size([64, 64, 32])


# Structural model

In [31]:
rotations = torch.eye(3).unsqueeze(0).expand(L, -1, -1).clone().to(device)
translations = torch.zeros(L, 3, device=device)

In [32]:
translations.shape

torch.Size([64, 3])

In [33]:
def apply_frame(rotations, translations, points):
    # R @ p + t - rotate then translate

    return torch.einsum('l i j, l j -> l i', rotations, points)+translations

def invert_frame(rotations, translations, points):
    # subtract translation then apply transpose of rotation (R^T = R^-1 for rotation matrices)
    return torch.einsum('l i j, l j -> l i', rotations.transpose(-1, -2), points - translations)


test_points = torch.randn(L, 3, device=device)
recovered = invert_frame(rotations, translations, apply_frame(rotations, translations, test_points))
print("Max roundtrip error:", (test_points-recovered).abs().max().item())

Max roundtrip error: 0.0


In [34]:
rotations.shape

torch.Size([64, 3, 3])

In [35]:
c_s = 128
# extract out the query sequence from Evoformer

s_proj = nn.Linear(c_m, c_s).to(device)
s = s_proj(m_out[0])
print("s:", s.shape) # L, c_s

s: torch.Size([64, 128])


# IPA

In [36]:
c_ipa = 16
num_heads_ipa = 4
num_qk_points = 4 # number of 3D Q/K points per head
num_v_points = 4 # number of 3D V points per head

# attention
ipa_to_q = nn.Linear(c_s, c_ipa * num_heads_ipa, bias=False).to(device)
ipa_to_k = nn.Linear(c_s, c_ipa * num_heads_ipa, bias=False).to(device)
ipa_to_v = nn.Linear(c_s, c_ipa * num_heads_ipa, bias=False).to(device)

# 3D points Q and K, each residue predics the num_qk_points points in its local frame
ipa_to_qp = nn.Linear(c_s, num_heads_ipa * num_qk_points * 3, bias=False).to(device)
ipa_to_kp = nn.Linear(c_s, num_heads_ipa * num_qk_points * 3, bias=False).to(device)
ipa_to_vp = nn.Linear(c_s, num_heads_ipa * num_v_points * 3, bias=False).to(device)

# pair bias from z
ipa_to_b = nn.Linear(c_z, num_heads_ipa, bias=False).to(device)

# project s to get QKV
q = rearrange(ipa_to_q(s), 'l (h d) -> h l d', h = num_heads_ipa)
k = rearrange(ipa_to_k(s), 'l (h d) -> h l d', h=num_heads_ipa)
v = rearrange(ipa_to_v(s), 'l (h d) -> h l d', h=num_heads_ipa)

# 3D points: each residue predicts point coords in its own local frame
qp = rearrange(ipa_to_qp(s), 'l (h p c) -> h l p c', h=num_heads_ipa, p=num_qk_points)
kp = rearrange(ipa_to_kp(s), 'l (h p c) -> h l p c', h=num_heads_ipa, p=num_qk_points)
vp = rearrange(ipa_to_vp(s), 'l (h p c) -> h l p c', h=num_heads_ipa, p=num_v_points)

print('q:', q.shape)
print('qp:',qp.shape)

q: torch.Size([4, 64, 16])
qp: torch.Size([4, 64, 4, 3])


In [37]:
def transform_points(rotations, translations, points):
    """
    rotations: L, 3, 3
    translations, L, 3
    point: H L P 3
    returns: H L P 3 (global space)
    """
    return torch.einsum('l i j, h l p j -> h l p i', rotations, points) + translations[None, :, None, :]

qp_global = transform_points(rotations, translations, qp)
kp_global = transform_points(rotations, translations, kp)

print("qp_global:", qp_global.shape)


qp_global: torch.Size([4, 64, 4, 3])


In [38]:
pt_scores = (
    (qp_global ** 2).sum(dim=(-1, -2)).unsqueeze(-1) +
    (kp_global ** 2).sum(dim=(-1, -2)).unsqueeze(-2) -
    2 * torch.einsum('h i p c, h j p c -> h i j', qp_global, kp_global)
)
pt_scores = - 0.5 * pt_scores
print("pt_scores:", pt_scores.shape)

pt_scores: torch.Size([4, 64, 64])


In [45]:
sequence = torch.einsum('h i d, h j d -> h i j', q, k) * (c_ipa ** -0.5)

pair = rearrange(ipa_to_b(z_out), 'i j h -> h i j')

atten = sequence + pair + pt_scores
atten = torch.softmax(atten, dim=-1)

In [ ]:
# sequence component
out_seq = torch.einsum('h i j, h j d -> h i d', atten, v) # H L D

In [49]:
# 3d component
out_pts = torch.einsum('h i j, h j p c -> h i p c', atten, transform_points(rotations, translations, vp)) # (H L P 3)

In [53]:
# pair component
out_pair = torch.einsum('h i j, i j d -> h i d', atten, z_out)